# Precision, Recall and F-Scores

**Notebook 3 of 5.** The confusion matrix holds four counts. This notebook turns them into
the two questions a fraud team actually asks, and then into single scores that weigh the
two answers against each other:

- **Recall:** of all the frauds, how many did we catch?
- **Precision:** of all the alarms we raised, how many were real frauds?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, fbeta_score

The same steps as in notebook 01, in one cell: load, split, scale, and train a logistic
regression.

In [ ]:
df = pd.read_csv("data/transactions.csv")
X = df.drop(columns="fraud")
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("TN:", tn, " FP:", fp, " FN:", fn, " TP:", tp)

`.ravel()` flattens the 2 by 2 matrix into its four counts, row by row, which is why they
come out in the order TN, FP, FN, TP.

## 1. Recall and precision

Both are a share of true positives, **out of different totals**:

$$\text{recall} = \frac{TP}{TP + FN} \qquad\qquad \text{precision} = \frac{TP}{TP + FP}$$

- **Recall** divides by all actual frauds, the bottom row of the matrix. It falls with
  every fraud missed.
- **Precision** divides by all predicted frauds, the right column. It falls with every
  false alarm.

> **Exercise:**
>
> Compute the model's recall and precision from `tp`, `fp` and `fn`, and check both
> against `recall_score` and `precision_score`.

In [ ]:
# your code: recall and precision, by hand and from scikit-learn

<details><summary>
Click here to compare your answer...
</summary>

**Recall 0.789, precision 0.970.** In words: the model catches about four frauds in five,
and when it raises an alarm it is right 97 times in 100. The two numbers describe the
model far better than its 99.8 percent accuracy did.
</details>

## 2. One number for both: the F1-score

Comparing models is easier with one number. Averaging precision and recall seems natural,
but the ordinary average forgives too much. Take a model that raises a single alarm, which
is correct:

In [ ]:
# one alarm, and it was right: perfect precision, but only 1 of 123 frauds caught
precision_one, recall_one = 1.0, 1 / 123

print(f"ordinary average: {(precision_one + recall_one) / 2:.3f}")
print(f"harmonic mean:    {2 * precision_one * recall_one / (precision_one + recall_one):.3f}")

The ordinary average gives this nearly useless model 0.5. The **harmonic mean** stays
close to the smaller of the two numbers, so a model is only rewarded when **both** are
good. That harmonic mean is the **F1-score**:

$$F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

## 3. When one matters more: F-beta

F1 treats precision and recall as equally important. Notebook 02 showed they rarely are.
The **F-beta** score adds a dial, `beta`, that says **how many times more recall counts
than precision**:

$$F_\beta = \frac{(1 + \beta^2) \cdot \text{precision} \cdot \text{recall}}{\beta^2 \cdot \text{precision} + \text{recall}}$$

| Score | Weighs | Suits |
| :-- | :-- | :-- |
| **F2** | recall more, `beta=2` | disease screening, fraud: missing a case is worse |
| **F1** | both equally, `beta=1` | no strong reason to favor either |
| **F0.5** | precision more, `beta=0.5` | spam filtering: a false alarm is worse |

In scikit-learn, `fbeta_score(y_true, y_pred, beta=2)`.

> **Exercise:**
>
> Compute the model's F1, F2 and F0.5. Which of the three is highest, and why that one,
> given its precision and recall?

In [ ]:
# your code: F1, F2 and F0.5 for the model

<details><summary>
Click here to compare your answer...
</summary>

**F0.5 is highest (0.927), then F1 (0.870), then F2 (0.819).** The model's strength is
precision, 0.97, and its weakness recall, 0.79. F0.5 weighs the strength more, F2 the
weakness. If the bank's priority is catching fraud, F2 is the honest score, and it says
there is room to improve.
</details>

> **Exercise:**
>
> No code needed. Which score would you report for each of these?
>
> 1. A model that decides which patients get a follow-up test for a treatable cancer.
> 2. A model that moves emails to the spam folder.
> 3. A model that suggests which job applicants a recruiter reads first, where both kinds
>    of mistake cost about the same.

Your answers here:

<details><summary>
Click here to compare your answers...
</summary>

1. **F2**, or recall itself: a missed cancer is far worse than an extra test.
2. **F0.5**, or precision itself: a real email lost in the spam folder is worse than one
   more spam in the inbox.
3. **F1**: no reason to favor either side.
</details>

## Summary

| Metric | Question | Formula |
| :-- | :-- | :-- |
| Recall | of the actual positives, how many were found? | TP / (TP + FN) |
| Precision | of the predicted positives, how many were right? | TP / (TP + FP) |
| F1 | both at once, equally weighted | harmonic mean |
| F-beta | both at once, recall weighted `beta` times more | F2, F0.5 |

- Unlike accuracy, none of these counts the true negatives, which is exactly why they
  still work when the negatives are 99 percent of the data.
- **Choose the score from the cost of the two mistakes**, before looking at any results.

Next, in `04_thresholds_and_curves.ipynb`: every one of these numbers depends on a
threshold we never chose.